# Optimasi Model Forecasting PM2.5 dengan Walk-Forward Validation

Notebook ini digunakan untuk melanjutkan tahap modeling setelah baseline LightGBM selesai dibuat. Fokus utama notebook ini adalah membangun proses validasi yang lebih kuat, melakukan tuning hyperparameter, dan melatih model final untuk setiap horizon prediksi PM2.5.

Input utama notebook ini adalah tiga dataset hasil preprocessing modeling:

```text
dataset_h6.csv
dataset_h12.csv
dataset_h24.csv
```

Ketiga dataset tersebut digunakan untuk memprediksi PM2.5 pada tiga horizon berbeda:

| Dataset | Horizon Prediksi | Target |
|---|---:|---|
| `dataset_h6.csv` | 6 jam ke depan | `target_pm25_t_plus_6` |
| `dataset_h12.csv` | 12 jam ke depan | `target_pm25_t_plus_12` |
| `dataset_h24.csv` | 24 jam ke depan | `target_pm25_t_plus_24` |

Berbeda dari notebook baseline yang menggunakan satu kali time-based split, notebook ini menggunakan **walk-forward time-series validation**. Pendekatan ini lebih sesuai untuk data time series karena proses evaluasi dilakukan pada beberapa periode validasi yang berurutan secara waktu.

Secara umum, notebook ini menjawab pertanyaan:

> Setelah LightGBM baseline dibuat, kombinasi hyperparameter mana yang memberikan performa terbaik untuk masing-masing horizon prediksi PM2.5?

## Validasi Time-Series yang Lebih Realistis

Notebook ini menggunakan **walk-forward validation** dengan 4 fold. Strategi ini digunakan agar proses evaluasi lebih mendekati kondisi forecasting sebenarnya, yaitu model dilatih menggunakan data historis dan diuji pada periode waktu setelahnya.

Alur validasi yang digunakan dapat diringkas sebagai berikut:

```text
Dataset horizon
        ↓
Urutkan data berdasarkan datetime dan station_slug
        ↓
Ambil daftar waktu unik
        ↓
Bagi bagian akhir timeline menjadi 4 fold validasi
        ↓
Untuk setiap fold:
    - training memakai data sebelum validation
    - validation memakai periode waktu setelah training
    - diberi purge gap sesuai horizon
        ↓
Hitung MAE dan RMSE setiap fold
        ↓
Ambil rata-rata performa CV
```

Notebook juga menerapkan **purge gap** sesuai horizon prediksi. Purge gap berarti ada jarak antara akhir data training dan awal data validation.

| Horizon | Purge Gap |
|---:|---:|
| H6 | 6 jam |
| H12 | 12 jam |
| H24 | 24 jam |

Purge gap penting karena target prediksi dibuat dari nilai PM2.5 di masa depan. Tanpa gap, ada risiko data training terlalu dekat dengan periode validation sehingga evaluasi menjadi terlalu optimistis.

Dengan validasi ini, hasil tuning menjadi lebih kuat dibanding hanya menggunakan satu split train-validation.

## Mencari Hyperparameter Terbaik dengan Optuna

Setelah validation framework terbentuk, notebook menjalankan tuning hyperparameter menggunakan **Optuna**.

Optuna digunakan untuk mencari kombinasi hyperparameter LightGBM yang menghasilkan nilai **CV MAE** paling kecil. Proses tuning dilakukan secara terpisah untuk setiap horizon, sehingga H6, H12, dan H24 memiliki parameter terbaik masing-masing.

Konfigurasi tuning utama:

| Komponen | Nilai |
|---|---:|
| Jumlah fold CV | 4 fold |
| Jumlah trial Optuna | 30 trial |
| Objective metric | Mean CV MAE |
| Model | LightGBM Regressor |
| Seed | 42 |

Parameter yang dicari oleh Optuna meliputi:

| Parameter | Fungsi |
|---|---|
| `n_estimators` | Jumlah pohon boosting |
| `learning_rate` | Kecepatan pembelajaran model |
| `num_leaves` | Kompleksitas struktur daun pohon |
| `max_depth` | Kedalaman maksimum pohon |
| `min_child_samples` | Minimum sampel pada leaf |
| `subsample` | Proporsi baris yang digunakan tiap iterasi |
| `colsample_bytree` | Proporsi fitur yang digunakan tiap pohon |
| `reg_alpha` | Regularisasi L1 |
| `reg_lambda` | Regularisasi L2 |

Hasil tuning disimpan dalam bentuk JSON dan CSV agar bisa digunakan ulang tanpa harus menjalankan tuning dari awal.

## Dari Best Params ke Final Model

Setelah parameter terbaik ditemukan, notebook melatih model final untuk masing-masing horizon.

Model final yang dilatih adalah:

| Horizon | Model Final |
|---:|---|
| H6 | `final_lgbm_h6.pkl` |
| H12 | `final_lgbm_h12.pkl` |
| H24 | `final_lgbm_h24.pkl` |

Model final dilatih menggunakan seluruh dataset pada masing-masing horizon dengan best params hasil tuning. Tujuannya adalah agar model final memanfaatkan seluruh data historis yang tersedia sebelum digunakan pada tahap evaluasi lanjutan atau deployment.

Selain model `.pkl`, notebook ini juga menyimpan:

1. Validation framework.
2. Hasil CV per fold.
3. Best params per horizon.
4. Ringkasan best params.
5. Ringkasan model final.
6. Feature importance final.

Catatan penting: nilai `full_mae` dan `full_rmse` pada final training adalah metrik pada data penuh yang dipakai untuk training. Jadi, angka tersebut bukan metrik validasi utama. Metrik validasi utama tetap berasal dari hasil walk-forward CV.

## Menyiapkan Konfigurasi Optimasi Model

Cell ini menyiapkan library, konfigurasi eksperimen, path dataset, folder output, serta fungsi dasar untuk membaca dan menyiapkan data modeling.

Library utama yang digunakan adalah:

| Library | Fungsi |
|---|---|
| `json` | Menyimpan best params dalam format JSON |
| `pickle` | Menyimpan model final dalam format `.pkl` |
| `pathlib.Path` | Mengatur folder output |
| `numpy` | Operasi numerik |
| `pandas` | Membaca dan mengolah dataset |
| `optuna` | Hyperparameter tuning |
| `lightgbm` | Model LightGBM |
| `sklearn.metrics` | Menghitung MAE dan RMSE |

Konfigurasi utama notebook adalah:

| Parameter | Nilai | Fungsi |
|---|---:|---|
| `SEED` | 42 | Menjaga eksperimen lebih reproducible |
| `N_FOLDS` | 4 | Jumlah fold walk-forward CV |
| `N_TRIALS` | 30 | Jumlah percobaan tuning Optuna |
| `OUT_DIR` | `outputs_phase2/` | Folder penyimpanan hasil optimasi |

Dataset input didefinisikan dalam dictionary `DATASET_PATHS`.

| Horizon | File Dataset |
|---:|---|
| 6 | `dataset_h6.csv` |
| 12 | `dataset_h12.csv` |
| 24 | `dataset_h24.csv` |

### Fungsi `load_dataset()`

Fungsi ini digunakan untuk membaca dataset horizon, mengubah kolom `datetime` dan `date` menjadi format datetime, lalu mengurutkan data berdasarkan:

```text
datetime
station_slug
```

Pengurutan ini penting karena dataset bersifat time series. Semua proses validasi harus mengikuti urutan waktu.

### Fungsi `prepare_xy()`

Fungsi ini memisahkan fitur `X`, target `y`, dan daftar nama fitur `feature_cols`.

Beberapa kolom dikeluarkan dari fitur karena tidak sesuai untuk training langsung:

| Kolom | Alasan Dikeluarkan |
|---|---|
| Target horizon | Label yang harus diprediksi, tidak boleh menjadi fitur |
| `datetime` | Waktu mentah, bukan fitur numerik langsung |
| `date` | Tanggal mentah |
| `station_name` | Redundan dengan `station_slug` |
| `lokasi` | Kolom teks lokasi |
| `season_simple` | Kolom string musim |
| `pm25_raw` | Backup nilai PM2.5 asli sebelum imputasi |
| `pm25_clean_full` | Duplikasi PM2.5 final jika tersedia |

Kolom bertipe object yang masih tersisa diubah menjadi tipe `category` agar dapat diproses oleh LightGBM.

In [1]:
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import optuna
import lightgbm as lgb
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

SEED = 42
N_FOLDS = 4
N_TRIALS = 30

DATASET_PATHS = {
    6: "dataset_h6.csv",
    12: "dataset_h12.csv",
    24: "dataset_h24.csv",
}

OUT_DIR = Path("outputs_phase2")
OUT_DIR.mkdir(exist_ok=True)

def load_dataset(path):
    df = pd.read_csv(path)
    if "datetime" in df.columns:
        df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.sort_values(["datetime", "station_slug"]).reset_index(drop=True)
    return df

def prepare_xy(df, target_col):
    exclude_cols = {
        target_col,
        "datetime",
        "date",
        "station_name",
        "lokasi",
        "season_simple",      # string
        "pm25_raw",           # backup raw
        "pm25_clean_full",    # duplikat dengan pm25 final
    }
    exclude_cols = {c for c in exclude_cols if c in df.columns}

    feature_cols = [c for c in df.columns if c not in exclude_cols]
    X = df[feature_cols].copy()
    y = df[target_col].copy()

    # object -> category untuk LightGBM
    for col in X.columns:
        if X[col].dtype == "object":
            X[col] = X[col].astype("category")

    return X, y, feature_cols

## Membentuk Walk-Forward Validation Framework

Cell ini membuat fungsi `make_walk_forward_folds()` untuk membentuk validation framework berbasis waktu.

Strategi yang digunakan adalah **expanding-window cross-validation**. Artinya, semakin maju fold validasi, semakin banyak data historis yang masuk ke training.

Setiap fold memiliki:

| Komponen | Makna |
|---|---|
| `fold_id` | Nomor fold |
| `train_end` | Batas akhir data training |
| `valid_start` | Awal periode validation |
| `valid_end` | Akhir periode validation |
| `train_idx` | Index baris training |
| `valid_idx` | Index baris validation |
| `n_train` | Jumlah baris training |
| `n_valid` | Jumlah baris validation |

### Validation Framework H6

| Fold | Train End | Valid Start | Valid End | n_train | n_valid |
|---:|---|---|---|---:|---:|
| 1 | `2024-11-18 00:00:00` | `2024-11-18 06:00:00` | `2025-03-28 02:00:00` | 93.480 | 15.585 |
| 2 | `2025-03-27 21:00:00` | `2025-03-28 03:00:00` | `2025-08-04 23:00:00` | 109.065 | 15.585 |
| 3 | `2025-08-04 18:00:00` | `2025-08-05 00:00:00` | `2025-12-12 20:00:00` | 124.650 | 15.585 |
| 4 | `2025-12-12 15:00:00` | `2025-12-12 21:00:00` | `2026-04-21 17:00:00` | 140.235 | 15.585 |

### Validation Framework H12

| Fold | Train End | Valid Start | Valid End | n_train | n_valid |
|---:|---|---|---|---:|---:|
| 1 | `2024-11-17 16:00:00` | `2024-11-18 04:00:00` | `2025-03-27 23:00:00` | 93.440 | 15.580 |
| 2 | `2025-03-27 12:00:00` | `2025-03-28 00:00:00` | `2025-08-04 19:00:00` | 109.020 | 15.580 |
| 3 | `2025-08-04 08:00:00` | `2025-08-04 20:00:00` | `2025-12-12 15:00:00` | 124.600 | 15.580 |
| 4 | `2025-12-12 04:00:00` | `2025-12-12 16:00:00` | `2026-04-21 11:00:00` | 140.180 | 15.580 |

### Validation Framework H24

| Fold | Train End | Valid Start | Valid End | n_train | n_valid |
|---:|---|---|---|---:|---:|
| 1 | `2024-11-16 20:00:00` | `2024-11-17 20:00:00` | `2025-03-27 14:00:00` | 93.340 | 15.575 |
| 2 | `2025-03-26 15:00:00` | `2025-03-27 15:00:00` | `2025-08-04 09:00:00` | 108.915 | 15.575 |
| 3 | `2025-08-03 10:00:00` | `2025-08-04 10:00:00` | `2025-12-12 04:00:00` | 124.490 | 15.575 |
| 4 | `2025-12-11 05:00:00` | `2025-12-12 05:00:00` | `2026-04-20 23:00:00` | 140.065 | 15.575 |

Output validation framework disimpan sebagai:

```text
outputs_phase2/validation_framework_4folds.csv
```

File ini penting karena menjadi dokumentasi resmi periode train-validation yang digunakan pada proses tuning.

In [2]:
def make_walk_forward_folds(df, horizon, n_folds=4, valid_ratio_each=0.10):
    """
    Expanding-window CV dengan purge gap sesuai horizon.
    - setiap fold valid diambil berurutan dari bagian akhir timeline
    - gap = horizon jam
    """
    df = df.sort_values(["datetime", "station_slug"]).reset_index(drop=True)
    unique_times = pd.Series(sorted(df["datetime"].dropna().unique()))

    n_times = len(unique_times)
    valid_size = max(1, int(n_times * valid_ratio_each))
    total_valid = n_folds * valid_size

    if total_valid >= n_times:
        raise ValueError("Terlalu banyak fold / valid_ratio terlalu besar untuk jumlah timestamp yang tersedia.")

    first_valid_start_idx = n_times - total_valid
    folds = []

    for fold_id in range(n_folds):
        valid_start_idx = first_valid_start_idx + fold_id * valid_size
        valid_end_idx = min(valid_start_idx + valid_size - 1, n_times - 1)

        valid_start = pd.Timestamp(unique_times.iloc[valid_start_idx])
        valid_end = pd.Timestamp(unique_times.iloc[valid_end_idx])

        # purge gap supaya label train tidak menyentuh periode valid
        train_end = valid_start - pd.Timedelta(hours=horizon)

        train_mask = df["datetime"] < train_end
        valid_mask = (df["datetime"] >= valid_start) & (df["datetime"] <= valid_end)

        train_idx = df.index[train_mask].to_numpy()
        valid_idx = df.index[valid_mask].to_numpy()

        folds.append({
            "fold_id": fold_id + 1,
            "train_end": train_end,
            "valid_start": valid_start,
            "valid_end": valid_end,
            "train_idx": train_idx,
            "valid_idx": valid_idx,
            "n_train": len(train_idx),
            "n_valid": len(valid_idx),
        })

    return folds

# buat dan simpan validation framework untuk semua horizon
all_fold_rows = []

for horizon, path in DATASET_PATHS.items():
    df_h = load_dataset(path)
    folds = make_walk_forward_folds(df_h, horizon=horizon, n_folds=N_FOLDS, valid_ratio_each=0.10)

    print(f"\n===== VALIDATION FRAMEWORK H{horizon} =====")
    for f in folds:
        row = {
            "horizon": horizon,
            "fold_id": f["fold_id"],
            "train_end": f["train_end"],
            "valid_start": f["valid_start"],
            "valid_end": f["valid_end"],
            "n_train": f["n_train"],
            "n_valid": f["n_valid"],
        }
        all_fold_rows.append(row)
        print(row)

validation_framework_df = pd.DataFrame(all_fold_rows)
validation_framework_df.to_csv(OUT_DIR / "validation_framework_4folds.csv", index=False)

print("\nSaved:", OUT_DIR / "validation_framework_4folds.csv")
validation_framework_df


===== VALIDATION FRAMEWORK H6 =====
{'horizon': 6, 'fold_id': 1, 'train_end': Timestamp('2024-11-18 00:00:00'), 'valid_start': Timestamp('2024-11-18 06:00:00'), 'valid_end': Timestamp('2025-03-28 02:00:00'), 'n_train': 93480, 'n_valid': 15585}
{'horizon': 6, 'fold_id': 2, 'train_end': Timestamp('2025-03-27 21:00:00'), 'valid_start': Timestamp('2025-03-28 03:00:00'), 'valid_end': Timestamp('2025-08-04 23:00:00'), 'n_train': 109065, 'n_valid': 15585}
{'horizon': 6, 'fold_id': 3, 'train_end': Timestamp('2025-08-04 18:00:00'), 'valid_start': Timestamp('2025-08-05 00:00:00'), 'valid_end': Timestamp('2025-12-12 20:00:00'), 'n_train': 124650, 'n_valid': 15585}
{'horizon': 6, 'fold_id': 4, 'train_end': Timestamp('2025-12-12 15:00:00'), 'valid_start': Timestamp('2025-12-12 21:00:00'), 'valid_end': Timestamp('2026-04-21 17:00:00'), 'n_train': 140235, 'n_valid': 15585}

===== VALIDATION FRAMEWORK H12 =====
{'horizon': 12, 'fold_id': 1, 'train_end': Timestamp('2024-11-17 16:00:00'), 'valid_start'

,horizon,fold_id,train_end,valid_start,valid_end,n_train,n_valid
0,6,1,2024-11-18 00:00:00,2024-11-18 06:00:00,2025-03-28 02:00:00,93480,15585
1,6,2,2025-03-27 21:00:00,2025-03-28 03:00:00,2025-08-04 23:00:00,109065,15585
2,6,3,2025-08-04 18:00:00,2025-08-05 00:00:00,2025-12-12 20:00:00,124650,15585
3,6,4,2025-12-12 15:00:00,2025-12-12 21:00:00,2026-04-21 17:00:00,140235,15585
4,12,1,2024-11-17 16:00:00,2024-11-18 04:00:00,2025-03-27 23:00:00,93440,15580
5,12,2,2025-03-27 12:00:00,2025-03-28 00:00:00,2025-08-04 19:00:00,109020,15580
6,12,3,2025-08-04 08:00:00,2025-08-04 20:00:00,2025-12-12 15:00:00,124600,15580
7,12,4,2025-12-12 04:00:00,2025-12-12 16:00:00,2026-04-21 11:00:00,140180,15580
8,24,1,2024-11-16 20:00:00,2024-11-17 20:00:00,2025-03-27 14:00:00,93340,15575
9,24,2,2025-03-26 15:00:00,2025-03-27 15:00:00,2025-08-04 09:00:00,108915,15575


## Menyusun Fungsi Evaluasi CV dan Objective Optuna

Cell ini mendefinisikan dua fungsi utama untuk proses tuning, yaitu `evaluate_lgbm_cv()` dan `make_optuna_objective()`.

### Fungsi `evaluate_lgbm_cv()`

Fungsi ini digunakan untuk mengevaluasi LightGBM menggunakan 4-fold walk-forward CV.

Tahapan di dalam fungsi ini adalah:

1. Memisahkan fitur dan target menggunakan `prepare_xy()`.
2. Membuat fold validasi menggunakan `make_walk_forward_folds()`.
3. Melatih LightGBM pada setiap fold.
4. Melakukan prediksi pada validation fold.
5. Menghitung MAE dan RMSE.
6. Menyimpan hasil evaluasi setiap fold ke dalam DataFrame.
7. Mengembalikan rata-rata MAE dari seluruh fold.

Metrik utama yang dikembalikan adalah:

```text
mean_mae
```

Nilai inilah yang digunakan Optuna untuk menentukan apakah satu kombinasi parameter lebih baik daripada kombinasi lainnya.

### Fungsi `make_optuna_objective()`

Fungsi ini membuat objective function untuk Optuna.

Pada setiap trial, Optuna mencoba kombinasi parameter LightGBM berikut:

| Parameter | Ruang Pencarian |
|---|---|
| `n_estimators` | 200 sampai 800 |
| `learning_rate` | 0.01 sampai 0.10, skala log |
| `num_leaves` | 15 sampai 127 |
| `max_depth` | 3 sampai 10 |
| `min_child_samples` | 10 sampai 100 |
| `subsample` | 0.7 sampai 1.0 |
| `colsample_bytree` | 0.7 sampai 1.0 |
| `reg_alpha` | 1e-8 sampai 10, skala log |
| `reg_lambda` | 1e-8 sampai 10, skala log |

Setiap kombinasi parameter dinilai menggunakan rata-rata MAE dari seluruh fold. Karena study Optuna dibuat dengan `direction="minimize"`, parameter terbaik adalah kombinasi dengan mean CV MAE paling kecil.

In [3]:
def evaluate_lgbm_cv(df, target_col, horizon, params, n_folds=4):
    """
    Evaluasi LightGBM dengan 4-fold walk-forward CV.
    Return:
    - mean_mae
    - detail per fold
    """
    X, y, feature_cols = prepare_xy(df, target_col)
    folds = make_walk_forward_folds(df, horizon=horizon, n_folds=n_folds, valid_ratio_each=0.10)

    fold_rows = []

    for f in folds:
        X_train = X.iloc[f["train_idx"]].copy()
        y_train = y.iloc[f["train_idx"]].copy()
        X_valid = X.iloc[f["valid_idx"]].copy()
        y_valid = y.iloc[f["valid_idx"]].copy()

        model = LGBMRegressor(
            random_state=SEED,
            verbose=-1,
            **params
        )

        model.fit(X_train, y_train)

        pred_valid = model.predict(X_valid)

        mae = mean_absolute_error(y_valid, pred_valid)
        rmse = np.sqrt(mean_squared_error(y_valid, pred_valid))

        fold_rows.append({
            "fold_id": f["fold_id"],
            "train_end": f["train_end"],
            "valid_start": f["valid_start"],
            "valid_end": f["valid_end"],
            "n_train": len(X_train),
            "n_valid": len(X_valid),
            "mae": mae,
            "rmse": rmse,
        })

    fold_df = pd.DataFrame(fold_rows)
    mean_mae = fold_df["mae"].mean()
    return mean_mae, fold_df, feature_cols


def make_optuna_objective(df, target_col, horizon):
    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 200, 800),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.10, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 15, 127),
            "max_depth": trial.suggest_int("max_depth", 3, 10),
            "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
            "subsample": trial.suggest_float("subsample", 0.7, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        }

        mean_mae, _, _ = evaluate_lgbm_cv(
            df=df,
            target_col=target_col,
            horizon=horizon,
            params=params,
            n_folds=N_FOLDS
        )
        return mean_mae

    return objective

## Menjalankan Tuning LightGBM untuk H6, H12, dan H24

Cell ini menjalankan tuning hyperparameter LightGBM untuk tiga horizon prediksi.

Proses tuning dilakukan dengan loop untuk setiap horizon:

1. Membaca dataset horizon.
2. Menentukan target horizon.
3. Membuat Optuna study dengan objective minimize.
4. Menjalankan 30 trial.
5. Menyimpan best params dan best CV MAE.
6. Mengevaluasi ulang best params untuk mendapatkan detail hasil per fold.
7. Menyimpan hasil tuning ke folder `outputs_phase2/`.

### Output Tuning

Untuk setiap horizon, notebook menyimpan:

| Output | Format File |
|---|---|
| Hasil CV per horizon | `cv_results_h{horizon}.csv` |
| Best params per horizon | `best_params_h{horizon}.json` |

Selain itu, notebook juga menyimpan gabungan semua horizon:

```text
outputs_phase2/cv_results_all_horizons.csv
outputs_phase2/best_params_all_horizons.json
```

### Best CV MAE

Berdasarkan output tuning, hasil terbaik setiap horizon adalah:

| Horizon | Best CV MAE |
|---:|---:|
| H6 | 4.2197 |
| H12 | 7.0104 |
| H24 | 10.8555 |

Hasil ini menunjukkan pola yang konsisten dengan baseline sebelumnya: semakin panjang horizon prediksi, semakin besar error model.

### Interpretasi

| Pola | Makna |
|---|---|
| H6 memiliki CV MAE paling rendah | Prediksi 6 jam ke depan masih sangat terbantu oleh informasi PM2.5 terbaru |
| H12 lebih sulit dari H6 | Model harus menangkap pola setengah harian |
| H24 paling sulit | Prediksi satu hari ke depan membutuhkan pola temporal dan cuaca yang lebih kompleks |
| Tuning dilakukan terpisah per horizon | Setiap horizon mendapat parameter yang sesuai dengan karakter datanya |

Cell ini adalah bagian paling berat secara komputasi karena setiap trial Optuna menjalankan 4-fold CV. Dengan 30 trial untuk 3 horizon, total proses training yang dilakukan cukup banyak.

In [4]:
best_params_all = {}
cv_summary_all = []

for horizon, path in DATASET_PATHS.items():
    print(f"\n{'='*20} TUNING H{horizon} {'='*20}")
    df_h = load_dataset(path)
    target_col = f"target_pm25_t_plus_{horizon}"

    study = optuna.create_study(direction="minimize")
    study.optimize(
        make_optuna_objective(df_h, target_col, horizon),
        n_trials=N_TRIALS,
        show_progress_bar=True
    )

    best_params = study.best_params
    best_score = study.best_value

    best_params_all[horizon] = {
        "target_col": target_col,
        "best_params": best_params,
        "best_cv_mae": best_score,
    }

    # evaluasi ulang best params supaya dapat detail per fold
    best_mean_mae, best_fold_df, feature_cols = evaluate_lgbm_cv(
        df=df_h,
        target_col=target_col,
        horizon=horizon,
        params=best_params,
        n_folds=N_FOLDS
    )

    best_fold_df["horizon"] = horizon
    cv_summary_all.append(best_fold_df)

    best_fold_df.to_csv(OUT_DIR / f"cv_results_h{horizon}.csv", index=False)

    with open(OUT_DIR / f"best_params_h{horizon}.json", "w") as f:
        json.dump(best_params_all[horizon], f, indent=2, default=str)

    print(f"Best CV MAE H{horizon}: {best_score:.4f}")
    print("Best params:", best_params)

cv_summary_df = pd.concat(cv_summary_all, ignore_index=True)
cv_summary_df.to_csv(OUT_DIR / "cv_results_all_horizons.csv", index=False)

with open(OUT_DIR / "best_params_all_horizons.json", "w") as f:
    json.dump(best_params_all, f, indent=2, default=str)

print("\nSaved tuning outputs to:", OUT_DIR)
cv_summary_df.head()


==================== TUNING H6 ====================


[I 2026-04-23 18:45:29,672] A new study created in memory with name: no-name-b230410f-e4ae-4d7d-9371-5f52b551953b


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-04-23 19:18:04,029] Trial 0 finished with value: 4.686893062711441 and parameters: {'n_estimators': 784, 'learning_rate': 0.06674202794693, 'num_leaves': 90, 'max_depth': 9, 'min_child_samples': 75, 'subsample': 0.8696634273405032, 'colsample_bytree': 0.8375500634082695, 'reg_alpha': 0.00013236053823982122, 'reg_lambda': 0.011256684217771254}. Best is trial 0 with value: 4.686893062711441.
[I 2026-04-23 19:27:01,780] Trial 1 finished with value: 4.361860607582408 and parameters: {'n_estimators': 349, 'learning_rate': 0.015219198225268954, 'num_leaves': 121, 'max_depth': 4, 'min_child_samples': 92, 'subsample': 0.8331634274121875, 'colsample_bytree': 0.7187970334212649, 'reg_alpha': 9.941953459968148e-05, 'reg_lambda': 2.052790032602676e-05}. Best is trial 1 with value: 4.361860607582408.
[I 2026-04-23 19:27:59,304] Trial 2 finished with value: 4.44579838108181 and parameters: {'n_estimators': 267, 'learning_rate': 0.02839268000104692, 'num_leaves': 63, 'max_depth': 7, 'min_chil

[I 2026-04-23 21:27:16,324] A new study created in memory with name: no-name-7b160f17-b2e5-4de4-a38d-4a5f613958bf


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-04-23 21:29:01,912] Trial 0 finished with value: 12.306103277114898 and parameters: {'n_estimators': 674, 'learning_rate': 0.04613114337330905, 'num_leaves': 26, 'max_depth': 10, 'min_child_samples': 99, 'subsample': 0.7851629788457994, 'colsample_bytree': 0.7719098937674949, 'reg_alpha': 0.014550785413769775, 'reg_lambda': 5.639911507671765e-07}. Best is trial 0 with value: 12.306103277114898.
[I 2026-04-23 21:30:28,817] Trial 1 finished with value: 11.438985618720025 and parameters: {'n_estimators': 375, 'learning_rate': 0.021910469178182002, 'num_leaves': 35, 'max_depth': 6, 'min_child_samples': 36, 'subsample': 0.9438385612707286, 'colsample_bytree': 0.9075299743162857, 'reg_alpha': 1.6929594105910867e-07, 'reg_lambda': 4.682467569841171e-05}. Best is trial 1 with value: 11.438985618720025.
[I 2026-04-23 21:31:42,901] Trial 2 finished with value: 11.413405011423093 and parameters: {'n_estimators': 204, 'learning_rate': 0.021648843375794327, 'num_leaves': 90, 'max_depth': 9,

,fold_id,train_end,valid_start,valid_end,n_train,n_valid,mae,rmse,horizon
0,1,2024-11-18 00:00:00,2024-11-18 06:00:00,2025-03-28 02:00:00,93480,15585,5.458874,18.911679,6
1,2,2025-03-27 21:00:00,2025-03-28 03:00:00,2025-08-04 23:00:00,109065,15585,4.197646,13.183140,6
2,3,2025-08-04 18:00:00,2025-08-05 00:00:00,2025-12-12 20:00:00,124650,15585,4.303065,8.743757,6
3,4,2025-12-12 15:00:00,2025-12-12 21:00:00,2026-04-21 17:00:00,140235,15585,2.919118,11.084959,6
4,1,2024-11-17 16:00:00,2024-11-18 04:00:00,2025-03-27 23:00:00,93440,15580,8.930783,23.722530,12


## Merangkum Parameter Terbaik per Horizon

Cell ini mengubah dictionary `best_params_all` menjadi tabel ringkas bernama `best_params_df`.

Tabel ini menyatukan informasi:

1. Horizon prediksi.
2. Best CV MAE.
3. Best hyperparameter LightGBM untuk horizon tersebut.

Hasil tabel disimpan sebagai:

```text
outputs_phase2/best_params_table.csv
```

### Best Params Hasil Tuning

| Horizon | Best CV MAE | n_estimators | learning_rate | num_leaves | max_depth | min_child_samples | subsample | colsample_bytree | reg_alpha | reg_lambda |
|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| H6 | 4.2197 | 703 | 0.017138 | 17 | 3 | 74 | 0.941154 | 0.834422 | 0.000002 | 0.001525 |
| H12 | 7.0104 | 543 | 0.022237 | 31 | 3 | 90 | 0.818127 | 0.972489 | 0.002065 | 0.000216 |
| H24 | 10.8555 | 417 | 0.010099 | 116 | 4 | 17 | 0.861558 | 0.712402 | 0.000177 | 0.000010 |

### Interpretasi Parameter

Beberapa hal yang dapat dibaca dari hasil tuning:

| Temuan | Interpretasi |
|---|---|
| H6 memilih `max_depth=3` dan `num_leaves=17` | Model H6 relatif sederhana, cocok untuk pola jangka pendek |
| H12 juga memilih `max_depth=3` | Prediksi 12 jam masih memperoleh manfaat dari model yang tidak terlalu dalam |
| H24 memilih `num_leaves=116` dan `max_depth=4` | Horizon 24 jam membutuhkan struktur pohon yang lebih kompleks |
| Learning rate H24 paling kecil | Model H24 belajar lebih pelan agar prediksi lebih stabil |
| Best params berbeda antarhorizon | Setiap horizon memiliki karakter prediksi yang berbeda |

Tabel ini menjadi acuan utama untuk final training.

In [5]:
best_params_table = []

for horizon, obj in best_params_all.items():
    row = {"horizon": horizon, "best_cv_mae": obj["best_cv_mae"]}
    row.update(obj["best_params"])
    best_params_table.append(row)

best_params_df = pd.DataFrame(best_params_table).sort_values("horizon").reset_index(drop=True)
best_params_df.to_csv(OUT_DIR / "best_params_table.csv", index=False)

print("=== BEST PARAMS PER HORIZON ===")
display(best_params_df)

=== BEST PARAMS PER HORIZON ===


,horizon,best_cv_mae,n_estimators,learning_rate,num_leaves,max_depth,min_child_samples,subsample,colsample_bytree,reg_alpha,reg_lambda
0,6,4.219676,703,0.017138,17,3,74,0.941154,0.834422,0.000002,0.001525
1,12,7.010438,543,0.022237,31,3,90,0.818127,0.972489,0.002065,0.000216
2,24,10.855505,417,0.010099,116,4,17,0.861558,0.712402,0.000177,0.000010


## Melatih Model Final Menggunakan Best Params

Cell ini melatih model final LightGBM untuk setiap horizon menggunakan best params hasil tuning.

Berbeda dengan proses CV, final training dilakukan menggunakan seluruh data pada masing-masing dataset horizon. Tujuannya adalah agar model final memanfaatkan seluruh data historis yang tersedia.

Alur final training:

1. Membaca dataset horizon.
2. Menentukan target horizon.
3. Menyiapkan fitur dan target menggunakan `prepare_xy()`.
4. Mengambil best params dari `best_params_all`.
5. Melatih `LGBMRegressor` pada seluruh data.
6. Menghitung MAE dan RMSE pada data penuh.
7. Menyimpan model final `.pkl`.
8. Menyimpan feature importance.
9. Menyimpan ringkasan final model.

### Ringkasan Final Model

| Horizon | Target | n_rows | Full-data MAE | Full-data RMSE | Model |
|---:|---|---:|---:|---:|---|
| H6 | `target_pm25_t_plus_6` | 155.850 | 3.1860 | 6.6694 | `outputs_phase2/final_lgbm_h6.pkl` |
| H12 | `target_pm25_t_plus_12` | 155.820 | 5.2452 | 9.9837 | `outputs_phase2/final_lgbm_h12.pkl` |
| H24 | `target_pm25_t_plus_24` | 155.760 | 8.1204 | 13.9073 | `outputs_phase2/final_lgbm_h24.pkl` |

### Catatan Interpretasi

Nilai `full_mae` dan `full_rmse` bukan metrik validasi utama. Angka tersebut dihitung pada data yang sama dengan data training final. Oleh karena itu, nilainya wajar lebih rendah daripada CV MAE.

Metrik yang lebih tepat untuk menilai kemampuan generalisasi model adalah:

```text
best_cv_mae
```

Sedangkan `full_mae` dan `full_rmse` lebih cocok digunakan sebagai sanity check bahwa model final berhasil fit terhadap data penuh.

### Output yang Disimpan

Untuk setiap horizon, cell ini menyimpan:

| Output | Format File |
|---|---|
| Model final | `final_lgbm_h{horizon}.pkl` |
| Feature importance final | `final_feature_importance_h{horizon}.csv` |

Ringkasan seluruh model final disimpan sebagai:

```text
outputs_phase2/final_model_summary.csv
```

In [6]:
final_model_summary = []
final_feature_importances = []

for horizon, path in DATASET_PATHS.items():
    print(f"\n{'='*20} FINAL TRAIN H{horizon} {'='*20}")
    df_h = load_dataset(path)
    target_col = f"target_pm25_t_plus_{horizon}"

    X_full, y_full, feature_cols = prepare_xy(df_h, target_col)
    params = best_params_all[horizon]["best_params"]

    final_model = LGBMRegressor(
        random_state=SEED,
        verbose=-1,
        **params
    )

    final_model.fit(X_full, y_full)

    pred_full = final_model.predict(X_full)
    full_mae = mean_absolute_error(y_full, pred_full)
    full_rmse = np.sqrt(mean_squared_error(y_full, pred_full))

    # save model
    model_path = OUT_DIR / f"final_lgbm_h{horizon}.pkl"
    with open(model_path, "wb") as f:
        pickle.dump(final_model, f)

    # save feature importance
    fi = pd.DataFrame({
        "feature": feature_cols,
        "importance": final_model.feature_importances_,
        "horizon": horizon
    }).sort_values("importance", ascending=False)

    fi.to_csv(OUT_DIR / f"final_feature_importance_h{horizon}.csv", index=False)
    final_feature_importances.append(fi)

    final_model_summary.append({
        "horizon": horizon,
        "target_col": target_col,
        "n_rows": len(df_h),
        "full_mae": full_mae,
        "full_rmse": full_rmse,
        "model_path": str(model_path),
    })

    print("Saved:", model_path)
    print("Full-data MAE :", round(full_mae, 4))
    print("Full-data RMSE:", round(full_rmse, 4))

final_model_summary_df = pd.DataFrame(final_model_summary).sort_values("horizon").reset_index(drop=True)
final_model_summary_df.to_csv(OUT_DIR / "final_model_summary.csv", index=False)

print("\n=== FINAL MODEL SUMMARY ===")
display(final_model_summary_df)


==================== FINAL TRAIN H6 ====================
Saved: outputs_phase2/final_lgbm_h6.pkl
Full-data MAE : 3.186
Full-data RMSE: 6.6694

==================== FINAL TRAIN H12 ====================
Saved: outputs_phase2/final_lgbm_h12.pkl
Full-data MAE : 5.2452
Full-data RMSE: 9.9837

==================== FINAL TRAIN H24 ====================
Saved: outputs_phase2/final_lgbm_h24.pkl
Full-data MAE : 8.1204
Full-data RMSE: 13.9073

=== FINAL MODEL SUMMARY ===


,horizon,target_col,n_rows,full_mae,full_rmse,model_path
0,6,target_pm25_t_plus_6,155850,3.186001,6.669376,outputs_phase2/final_lgbm_h6.pkl
1,12,target_pm25_t_plus_12,155820,5.245217,9.983733,outputs_phase2/final_lgbm_h12.pkl
2,24,target_pm25_t_plus_24,155760,8.120361,13.907278,outputs_phase2/final_lgbm_h24.pkl


In [7]:
from pathlib import Path

for fp in sorted(OUT_DIR.glob("final_lgbm_h*.pkl")):
    print(fp.name, "| exists =", fp.exists(), "| size =", fp.stat().st_size, "bytes")

final_lgbm_h12.pkl | exists = True | size = 455513 bytes
final_lgbm_h24.pkl | exists = True | size = 668787 bytes
final_lgbm_h6.pkl | exists = True | size = 590033 bytes


# Struktur Folder dan File Output Notebook

Notebook ini menghasilkan satu folder output utama:

```text
outputs_phase2/
```

Folder ini menyimpan seluruh hasil validasi, tuning, final training, model `.pkl`, dan feature importance.

Input utama notebook adalah:

```text
dataset_h6.csv
dataset_h12.csv
dataset_h24.csv
```

Struktur output notebook dapat diringkas sebagai berikut:

```text
project/
│
├── dataset_h6.csv
├── dataset_h12.csv
├── dataset_h24.csv
│
└── outputs_phase2/
    ├── validation_framework_4folds.csv
    │
    ├── cv_results_h6.csv
    ├── cv_results_h12.csv
    ├── cv_results_h24.csv
    ├── cv_results_all_horizons.csv
    │
    ├── best_params_h6.json
    ├── best_params_h12.json
    ├── best_params_h24.json
    ├── best_params_all_horizons.json
    ├── best_params_table.csv
    │
    ├── final_lgbm_h6.pkl
    ├── final_lgbm_h12.pkl
    ├── final_lgbm_h24.pkl
    │
    ├── final_feature_importance_h6.csv
    ├── final_feature_importance_h12.csv
    ├── final_feature_importance_h24.csv
    │
    └── final_model_summary.csv
```

### Penjelasan File Output

| File | Fungsi |
|---|---|
| `validation_framework_4folds.csv` | Dokumentasi fold walk-forward validation untuk H6, H12, dan H24 |
| `cv_results_h6.csv` | Hasil CV per fold untuk horizon 6 jam |
| `cv_results_h12.csv` | Hasil CV per fold untuk horizon 12 jam |
| `cv_results_h24.csv` | Hasil CV per fold untuk horizon 24 jam |
| `cv_results_all_horizons.csv` | Gabungan hasil CV seluruh horizon |
| `best_params_h6.json` | Best params LightGBM untuk H6 |
| `best_params_h12.json` | Best params LightGBM untuk H12 |
| `best_params_h24.json` | Best params LightGBM untuk H24 |
| `best_params_all_horizons.json` | Gabungan best params seluruh horizon |
| `best_params_table.csv` | Tabel ringkas best params semua horizon |
| `final_lgbm_h6.pkl` | Model final LightGBM horizon 6 jam |
| `final_lgbm_h12.pkl` | Model final LightGBM horizon 12 jam |
| `final_lgbm_h24.pkl` | Model final LightGBM horizon 24 jam |
| `final_feature_importance_h6.csv` | Feature importance model final H6 |
| `final_feature_importance_h12.csv` | Feature importance model final H12 |
| `final_feature_importance_h24.csv` | Feature importance model final H24 |
| `final_model_summary.csv` | Ringkasan final model semua horizon |

### Ringkasan Hasil Tuning

| Horizon | Best CV MAE | Full-data MAE | Full-data RMSE |
|---:|---:|---:|---:|
| H6 | 4.2197 | 3.1860 | 6.6694 |
| H12 | 7.0104 | 5.2452 | 9.9837 |
| H24 | 10.8555 | 8.1204 | 13.9073 |

Secara keseluruhan, notebook ini berhasil melakukan optimasi LightGBM untuk tiga horizon prediksi PM2.5. Hasilnya menunjukkan bahwa horizon 6 jam tetap menjadi prediksi yang paling mudah, sedangkan horizon 24 jam menjadi prediksi paling sulit.

Output terpenting dari notebook ini adalah tiga model final berikut:

```text
outputs_phase2/final_lgbm_h6.pkl
outputs_phase2/final_lgbm_h12.pkl
outputs_phase2/final_lgbm_h24.pkl
```

Ketiga model tersebut dapat digunakan pada tahap berikutnya untuk evaluasi final, analisis feature importance, interpretasi model, atau inference pada data terbaru.